# Informer 时序预测模型

本 notebook 展示 Informer 模型的架构、训练流程和评估指标。

**核心创新**：
- **ProbSparse 自注意力**：O(L log L) 复杂度，只计算 Top-K 重要的 query
- **生成式解码器**：一次性生成所有预测步，而非自回归逐步解码
- **蒸馏层**：逐层压缩序列长度（当前实现保留完整序列长度）

In [ ]:
import sys
sys.path.insert(0, '../../')

import torch
import numpy as np
import matplotlib.pyplot as plt

from models import InformerModel, TimeSeriesDataset, Trainer
from models.trainer import resolve_device
from torch.utils.data import DataLoader, Subset
from pathlib import Path

device = resolve_device('auto')
print(f'Device: {device}')

DATA_DIR = Path('../../') / 'data' / 'processed'


## 1. 模型架构

```
输入 (batch, lookback, features)
  ↓ Linear(input_size, d_model) + PositionalEncoding
  ↓ × N InformerEncoderLayer
  │   ├─ ProbSparseAttention (O(L log L))
  │   │   ├─ 采样 factor×ln(L_K) 个 Key
  │   │   ├─ 计算 query 重要性，选 Top-K query
  │   │   └─ Top-K query 做完整注意力，其余用 V 均值填充
  │   └─ FFN (d_model → d_ff → d_model)
  ↓ GenerativeDecoder
  │   ├─ Linear(lookback, horizon) 时间投影
  │   └─ × M Refine Block (残差细化)
  ↓ Linear(d_model, input_size)
输出 (batch, horizon, features)
```

In [ ]:
ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
print(f'数据集: ETTh1 h96')
print(f'  input_size: {ds.input_size}, target_idx: {ds.target_idx}')
print(f'  训练样本: {len(ds)}, X: {ds.X.shape}, Y: {ds.Y.shape}')

# 默认配置 (与正式实验一致)
model = InformerModel(input_size=ds.input_size, d_model=64, n_heads=4,
                       n_encoder_layers=2, n_decoder_layers=1,
                       d_ff=128, factor=5, dropout=0.1, horizon=96)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n默认配置参数量: {params:,}')

# 前向传播
x = torch.randn(2, 96, ds.input_size)
y = model(x)
print(f'前向传播: {x.shape} → {y.shape}')

# ProbSparse 注意力可视化：看看哪些时间步被选为 Top-K
print(f'\nProbSparse factor=5, lookback=96')
print(f'  采样 Key 数: factor × ln(L_K) = 5 × ln(96) ≈ {5 * np.log(96):.0f}')
print(f'  Top-K query 数: factor × ln(L_Q) ≈ {5 * np.log(96):.0f} (同上)')
print(f'  完整注意力计算量: Top-K 个，其余用 V 均值填充')

## 2. 快速训练与评估

In [ ]:
train_ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
val_ds   = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'val')
test_ds  = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'test')

train_loader = DataLoader(Subset(train_ds, range(512)), batch_size=32, shuffle=True)
val_loader   = DataLoader(Subset(val_ds,   range(128)), batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

model = InformerModel(input_size=ds.input_size, d_model=64, n_heads=4,
                       n_encoder_layers=2, n_decoder_layers=1,
                       d_ff=128, factor=5, dropout=0.1, horizon=96)
print(f'参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

trainer = Trainer(model, device=device, lr=1e-3, weight_decay=1e-5, seed=42)
history = trainer.train(train_loader, val_loader, epochs=5, patience=10)
print(f'\n训练完成: {len(history["train_losses"])} epochs, best_val_loss={history["best_val_loss"]:.4f}')

## 3. 评估指标

In [ ]:
predictions, targets = trainer.predict(test_loader)
metrics = trainer.compute_metrics(predictions, targets, target_idx=ds.target_idx)

print('=== 全变量指标 ===')
print(f'  MSE:  {metrics["MSE"]:.4f}')
print(f'  MAE:  {metrics["MAE"]:.4f}')
print(f'  R²:   {metrics["R2"]:.4f}')
print(f'  MAPE: {metrics["MAPE"]:.2f}%')
print('\n=== 目标列指标 (OT) ===')
print(f'  MSE_target:  {metrics["MSE_target"]:.4f}')
print(f'  MAE_target:  {metrics["MAE_target"]:.4f}')
print(f'  R²_target:   {metrics["R2_target"]:.4f}')
print(f'  MAPE_target: {metrics["MAPE_target"]:.2f}%')

## 4. 可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(history['train_losses'], label='Train Loss')
ax.plot(history['val_losses'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Informer Training')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
pred_target = predictions[:64, :, ds.target_idx]
true_target = targets[:64, :, ds.target_idx]
ax.plot(true_target.flatten(), label='True', alpha=0.7)
ax.plot(pred_target.flatten(), label='Pred', alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Informer Prediction vs True (OT)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()